In [ ]:
import json
from pathlib import Path
import nbformat as nbf
from __future__ import annotations
import re
import time
from typing import List, Dict, Any, Iterable, Tuple, Optional
import requests

In [ ]:
OLLAMA_MODELS: Dict[str, Dict[str, Any]] = {
    # Multilíngue forte + contexto longo: ótimo p/ pt-BR e prompts longos
    "qwen2.5:14b-instruct-q4_K_M": {
        "options": {"num_ctx": 12000, "temperature": 0.2},
    },
    # Baseline rápido e estável; boa compatibilidade de prompts
    "llama3.1:8b-instruct-q4_K_M": {
        "options": {"num_ctx": 8000, "temperature": 0.2},
    },
    # MoE com ótimo trade-off qualidade/latência local
    "mixtral:8x7b-instruct-q4_K_M": {
        "options": {"num_ctx": 8000, "temperature": 0.2},
    },
    # Leve e eficiente; boa em instruções gerais
    "mistral:7b-instruct-v0.3-q4_K_M": {
        "options": {"num_ctx": 8000, "temperature": 0.2},
    },
}


In [ ]:
iob_labels = (
        "O",
        "B-baciaSedimentar",
        "I-baciaSedimentar",
        "B-epoca", 
        "I-epoca",
        "B-idade",
        "I-idade",
        "B-periodo",
        "I-periodo",
        "B-eon",
        "I-eon",
        "B-era",
        "I-era",
        "B-magmaticas",
        "I-magmaticas",
        "B-metamorficas",
        "I-metamorficas",
        "B-sedimentaresSiliciclasticas",
        "I-sedimentaresSiliciclasticas",
        "B-sedimentaresCarbonaticas",
        "I-sedimentaresCarbonaticas",
        "B-unidadeEstratigrafica",
        "I-unidadeEstratigrafica",
        "B-contextoGeologicoDeBacia",
        "I-contextoGeologicoDeBacia",
        "B-ambienteSedimentacao",
        "I-ambienteSedimentacao",
        "B-constituinteRochaSedimentar",
        "I-constituinteRochaSedimentar",
        "B-fosseis",
        "I-fosseis",
        "B-planctonico",
        "I-planctonico",
        "B-bentonico",
        "I-bentonico",
        "B-mineral",
        "I-mineral",
        "B-procedimentoMetodologico",
        "I-procedimentoMetodologico"
    )

In [ ]:
def tokens_to_text(tokens: List[str]) -> str:
    # Junta preservando espaços simples.
    return " ".join(tokens)

def fmt_example(ex: Dict[str, Any]) -> str:
    # Espera ex = {"tokens": [...], "tags": [...]}, 1-based no output.
    toks = ex["tokens"]
    tags = ex["tags"]
    pairs = [f"{i+1}-{t}" for i, t in enumerate(tags)]
    return f"Tokens: {tokens_to_text(toks)}\nSaída: {' '.join(pairs)}"

def build_prompt(few_shot: List[Dict[str, Any]], query_tokens: List[str]) -> str:
    N = len(query_tokens)
    header = f"""
        Você é um anotador especialista em NER. Rotule **cada token** usando o esquema **IOB2**.

        Regras (case-sensitive):
        1) Use apenas os rótulos:\n{IOB_LABELS}

        2) IOB2 (cheque silenciosamente):
        • Uma entidade inicia em B-<TIPO>.
        • I-<TIPO> só após B-<TIPO> ou I-<TIPO> do mesmo TIPO.
        • Nunca iniciar com I-.

        3) Formato ÚNICO de saída (sem texto extra):
        • Uma única linha com N pares índice-rótulo (1-based), separados por espaço.
        • Cada par: <índice>-<RÓTULO>.
        • Ex.: 1-O 2-B-magmaticas 3-I-magmaticas ... N-O

        4) Se houver N tokens, produza exatamente N rótulos.
        """.strip()

            examples = "\n\n### Exemplos\n" + "\n\n".join(fmt_example(ex) for ex in few_shot)

            task = f"""
        ### Tarefa
        Tokens: {tokens_to_text(query_tokens)}

        ### Saída esperada
        Você tem {N} tokens. Produza exatamente {N} pares no formato índice-rótulo, em uma única linha, e nada mais.
        """.strip()

    return f"{header}\n\n{examples}\n\n{task}"

In [ ]:
def ollama_generate(
    model: str,
    prompt: str,
    options: Optional[Dict[str, Any]] = None,
    host: str = "http://localhost:11434",
    timeout: int = 120,
    stream: bool = False,
) -> str:
    """
    Wrapper do endpoint /api/generate.
    """
    url = f"{host}/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": stream,
        "options": options or {},
        # "stop": ["\n\n"],  # ajuste opcional se algum modelo insistir em linhas extras
    }
    # Timeout levemente alto pois alguns modelos podem demorar.
    resp = requests.post(url, json=payload, timeout=timeout)
    resp.raise_for_status()

    if stream:
        # Se usar stream=True, agregamos as linhas 'data: {...}'
        text = ""
        for line in resp.iter_lines():
            if not line:
                continue
            try:
                obj = json.loads(line.decode("utf-8"))
            except Exception:
                continue
            text += obj.get("response", "")
        return text
    else:
        data = resp.json()
        return data.get("response", "")

In [ ]:
def parse_indexed_labels(text: str, N: int) -> List[str]:
    """
    Aceita '1-O 2-B-foo 3-I-foo ...' e retorna lista de rótulos com tamanho N.
    Se faltar algum índice, preenche com 'O'.
    """
    labels = ["O"] * N
    for m in PAIR_RE.finditer(text.strip()):
        idx = int(m.group(1))
        lab = m.group(2)
        if 1 <= idx <= N:
            labels[idx - 1] = lab
    return labels

In [ ]:
def predict_labels_ollama(
    model_key: str,
    few_shot: List[Dict[str, Any]],
    query_tokens: List[str],
    host: str = "http://localhost:11434",
) -> List[str]:
    cfg = OLLAMA_MODELS[model_key]
    prompt = build_prompt(few_shot, query_tokens)
    raw = ollama_generate(model=model_key, prompt=prompt, options=cfg.get("options"), host=host, stream=False)
    return parse_indexed_labels(raw, N=len(query_tokens))

def batch_predict(
    model_key: str,
    few_shot: List[Dict[str, Any]],
    batch_sentences: List[List[str]],
    host: str = "http://localhost:11434",
    sleep_s: float = 0.0,
) -> List[List[str]]:
    """
    Aplica predict_labels_ollama para um lote de sentenças.
    sleep_s pode ajudar a evitar sobrecarga do servidor local.
    """
    out = []
    for toks in batch_sentences:
        out.append(predict_labels_ollama(model_key, few_shot, toks, host=host))
        if sleep_s > 0:
            time.sleep(sleep_s)
    return out

def available_models() -> List[str]:
    return list(OLLAMA_MODELS.keys())